# Previsão de Household Income por município (até 2026)

**Objetivo**: a partir do ficheiro `household-income-15-21.parquet`, projetar valores de *household income* para 2022–2026, município a município, usando 5 componentes: (1) tendência local (CAGR pré-COVID), (2) inflação observada, (3) crescimento do PIB, (4) ajuste territorial (cluster+convergência) e (5) crescimento salarial (mínimo + proxy mediano).

**Nota**: este notebook foi gerado em 2026-05-16.

## Fontes (para defender o trabalho)

Não é feito forecast da inflação/salário mínimo; são usadas séries publicadas.

- Salário mínimo Mensal nacional (tabela anual, inclui 2026): https://www.pordata.pt/portugal/evolucao+do+salario+minimo+nacional-74

- Salário Médio Líquido Mensal Em portugal - https://pt.tradingeconomics.com/portugal/wages

- Previsão macro (PIB) – PORDATA - https://www.pordata.pt/portugal/taxa+de+crescimento+do+pib-2298
    - Estimativa para 2026 https://www.bportugal.pt/publicacao/boletim-economico-marco-2026

-Inflação segundo Banco de Portugal - https://bpstat.bportugal.pt/serie/12704650
    - Estimativa para 2026: https://pt.tradingeconomics.com/portugal/inflation-cpi


- (Proxy para mediana) ‘Rendimento mediano disponível’ na PORDATA Retratos da Europa: https://retratos.europa.pordata.pt/custo-de-vida/portugal


In [11]:
# Imports
import pandas as pd
import numpy as np


In [12]:
df = pd.read_parquet('household-income-15-21.parquet')

df

,ANO,Município,Household Income (EUR)
0,2015,Abrantes,15108
1,2015,Aguiar da Beira,11334
2,2015,Alandroal,11515
3,2015,Albergaria-a-Velha,14776
4,2015,Albufeira,13363
...,...,...,...
2354,2021,Área Metropolitana de Lisboa,23321
2355,2021,Área Metropolitana do Porto,20110
2356,2021,Évora,22087
2357,2021,Ílhavo,20196


## 0) Análise de municípios: Ler o Dataset e normalizar colunas

De parquet para dataframe e normalização de colunas para evitar erros no pipeline.

In [13]:
# De parquet para dataframe.
df = pd.read_parquet('household-income-15-21.parquet')



# Mapeamento de nomes alternativos para year/municipio/income.
rename_map = {
    'ANO': 'year',
    'Município': 'municipio',
    'Household Income (EUR)': 'income',
}

# Intera-se entre pares do rename_map para renomear colunas correspondentes.
for k, v in rename_map.items():
    if k in df.columns:
        df = df.rename(columns={k: v})

# Conversão de tipos.
df['year'] = df['year'].astype(int)
df['municipio'] = df['municipio'].astype(str)
df['income'] = pd.to_numeric(df['income'], errors='coerce')

# Ordenação por município e ano para cálculo temporal subsequente.
df = df.sort_values(['municipio', 'year']).reset_index(drop=True)

df.head(8)

,year,municipio,income
0,2015,Abrantes,15108
1,2016,Abrantes,15609
2,2017,Abrantes,16014
3,2018,Abrantes,16574
4,2019,Abrantes,17115
5,2020,Abrantes,17326
6,2021,Abrantes,18005
7,2015,Aguiar da Beira,11334


## 1) Calcular CAGR pré-COVID (até 2019)

CAGR significa Compound Annual Growth Rate. O objetivo é calcular o crescimento anual composto de cada município até 2019, evitando o efeito do choque COVID.

A lógica é fixar o município e medir a variação entre os pontos inicial e final do período pré-COVID. Por exemplo, se o income passa de 100 para 121 em 2 anos, o CAGR é (121/100)^(1/2) - 1 = 0,10, ou 10% ao ano.

In [14]:
def calculate_pre_covid_cagr(group: pd.DataFrame) -> float:
    """Calculate CAGR for one municipio using observations up to 2019."""
    """
    Args:
        group: DataFrame slice for a single municipio (all years for that municipio).
               The caller uses `df.groupby('municipio')`, so each `group` contains only one municipio,
               e.g. rows for 'Abrantes' from 2015..2019. This function will further filter to years <= 2019.

    Returns:
        CAGR as float (e.g. 0.10 for 10%/year) or np.nan if not computable.
    """
    # Filtra até 2019 para estimar crescimento estrutural sem o efeito COVID.
    g = group[group['year'] <= 2019].copy()

    # São necessários pelo menos dois pontos no tempo para calcular CAGR.
    # Exemplo: para Abrantes pode haver linhas para 2015,2016,2017,2018,2019 -> shape[0] >= 2 ok.
    if g.shape[0] < 2:
        return np.nan

    # Pegamos o primeiro e o último valor observados no período filtrado (mesmo municipio).
    start = g.iloc[0]['income']
    end = g.iloc[-1]['income']
    # Número de anos entre os dois pontos (p.ex. 2019 - 2015 = 4).
    years = g.iloc[-1]['year'] - g.iloc[0]['year']



    # Fórmula do CAGR: (end/start)^(1/years) - 1.
    # Exemplo: start=100, end=121, years=2 => (121/100)^(1/2) - 1 = 0.10 (10% ao ano).
    return (end / start) ** (1 / years) - 1

cagr_df = (
    df.groupby('municipio', as_index=False)
      .apply(lambda g: pd.Series({'cagr_pre_covid': calculate_pre_covid_cagr(g)}))
)

cagr_df.head()

,municipio,cagr_pre_covid
0,Abrantes,0.031674
1,Aguiar da Beira,0.056820
2,Alandroal,0.048184
3,Albergaria-a-Velha,0.034377
4,Albufeira,0.034842


## 2) Tabela Macro com fatores económicos para extrapolar os Median Household Income (inflação, PIB, salários)

Dados com bases citadas acima.

- O ambiente do notebook pode não ter acesso à Internet, por isso as séries são definidas manualmente e validadas.
- É verificado que todos os anos necessários estão presentes para evitar buracos no modelo.

**Como preencher**: copiar os valores dos sites e substituir nos dicionários abaixo.

In [15]:
# ======== CONFIG (PREENCHER) ========
YEARS_FORECAST = [2022, 2023, 2024, 2025, 2026]

# (2) Inflação anual (% em decimal). Fonte recomendada: INE (IPC) / PORDATA / ECB.
# Exemplo: 2.4% -> 0.024
inflation = {
    2022: 0.0783,
    2023: 0.0431,
    2024: 0.0242,
    2025: 0.0234,
    2026: 0.0330,
}

# (3) Crescimento real do PIB (% em decimal). Fonte: Comissão Europeia / Banco de Portugal / OCDE.
gdp_growth = {
    2022: 0.07,
    2023: 0.0310,
    2024: 0.022,
    2025: 0.019,
    2026: 0.018,
}

# (5) Salário mínimo nacional (valor mensal, EUR). Fonte: PORDATA ou DGERT.
min_wage = {
    2022: 705,
    2023: 760,
    2024: 820,
    2025: 870,
    2026: 920,
}

# (5) Proxy mediano (preferido): rendimento mediano disponível (EUR/mês) ou salário mediano se houver.
# Fonte proxy (PORDATA retratos): rendimento mediano disponível.
median_proxy = {
    2022: 1010,
    2023: 1090,
    2024: 1230,
    2025: 1310,
    2026: 1370,
}

# Pesos para combinar mínimo e proxy mediano num único driver salarial.
# O proxy mediano recebe mais peso por estar mais alinhado com household income.
W_MIN = 0.30
W_MED = 0.70

# ======== FIM CONFIG ========

def _validate_years(d: dict, name: str, years=YEARS_FORECAST):
    missing = [y for y in years if y not in d]
    if missing:
        raise ValueError(f'Faltam anos em {name}: {missing}')
    nan_years = [y for y in years if pd.isna(d[y])]
    if nan_years:
        raise ValueError(
            f'Faltam valores (NaN) em {name} para anos: {nan_years}. ' 
            f'Preenche estes valores com base nas fontes do topo.'
        )

_validate_years(inflation, 'inflation')
_validate_years(gdp_growth, 'gdp_growth')
_validate_years(min_wage, 'min_wage')
_validate_years(median_proxy, 'median_proxy')

macro = pd.DataFrame({
    'year': YEARS_FORECAST,
    'inflation': [inflation[y] for y in YEARS_FORECAST],
    'gdp_growth': [gdp_growth[y] for y in YEARS_FORECAST],
    'min_wage': [min_wage[y] for y in YEARS_FORECAST],
    'median_proxy': [median_proxy[y] for y in YEARS_FORECAST],
})

# Conversão de salário mínimo e proxy mediano em taxas de crescimento anuais.
macro['g_min_wage'] = macro['min_wage'].pct_change()
macro['g_median_proxy'] = macro['median_proxy'].pct_change()

# Para 2022, não há variação anterior dentro do horizonte. Assume-se 0 para evitar salto artificial.
macro.loc[macro['year'] == YEARS_FORECAST[0], ['g_min_wage', 'g_median_proxy']] = 0.0

# Driver salarial agregado com pesos definidos.
macro['g_wage_signal'] = W_MIN * macro['g_min_wage'] + W_MED * macro['g_median_proxy']

macro

,year,inflation,gdp_growth,min_wage,median_proxy,g_min_wage,g_median_proxy,g_wage_signal
0,2022,0.0783,0.070,705,1010,0.000000,0.000000,0.000000
1,2023,0.0431,0.031,760,1090,0.078014,0.079208,0.078850
2,2024,0.0242,0.022,820,1230,0.078947,0.128440,0.113592
3,2025,0.0234,0.019,870,1310,0.060976,0.065041,0.063821
4,2026,0.0330,0.018,920,1370,0.057471,0.045802,0.049302


## 3) Ajuste territorial (cluster + convergência)

Definição de um ajuste territorial que combina o efeito de cluster urbano/extra-urbano com convergência suave ao rendimento médio nacional.

- Cluster urbano vs resto (ajuste leve)
- Convergência suave (municípios abaixo da média crescem % ligeiramente mais)

A abordagem é mantida simples para ser auditável e fácil de explicar.

## 4) Preparar base (último ano observado) e juntar features

Construção do estado inicial por município no último ano real observado, juntando CAGR pré-COVID e classificação de cluster.

In [16]:
latest_year = int(df['year'].max())

base_df = df[df['year'] == latest_year].copy()

base_df = base_df.merge(cagr_df, on='municipio', how='left')

base_df.head()

,year,municipio,income,cagr_pre_covid
0,2021,Abrantes,18005,0.031674
1,2021,Aguiar da Beira,15269,0.056820
2,2021,Alandroal,14850,0.048184
3,2021,Albergaria-a-Velha,18064,0.034377
4,2021,Albufeira,15233,0.034842


## 5) Loop de projeção (2022–2026) e dataset final

Projeção ano a ano aplicando as cinco componentes:

1) CAGR local (pré-COVID)
2) inflação
3) PIB
5) salários (mínimo + proxy mediano)

O histórico observacional é combinado com as projeções e o resultado é gravado em parquet.

In [17]:
projections = []
current_df = base_df.copy()

for _, row in macro.sort_values('year').iterrows():
    year = int(row['year'])

    # Intera-se pelos anos do macro dataset ordenado.
    national_avg = current_df['income'].mean()

    next_df = current_df.copy()

    # (1 + 4) Crescimento local ajustado por CAGR e cluster.
    next_df['g_local_adj'] = next_df.apply(
        lambda r: adjust_growth(
            base_growth=r['cagr_pre_covid'],
            cluster=r['cluster'],
            current_income=r['income'],
            national_avg=national_avg
        ),
        axis=1
    )

    # (2) inflação observada.
    g_infl = float(row['inflation'])

    # (3) PIB.
    g_gdp = float(row['gdp_growth'])

    # (5) driver salarial agregado.
    g_wage = float(row['g_wage_signal'])

    # Combinação de fatores em uma taxa total.
    # Exemplo: se g_local_adj=0.03, g_infl=0.02, g_gdp=0.018 e g_wage=0.05,
    # g_total = 0.03 + (2*0.02 + 0.018 + 7*0.05)/10 = 0.03 + 0.0418 = 0.0718.
    next_df['g_total'] = next_df['g_local_adj'] + (2*g_infl + g_gdp + 7*g_wage)/10

    # Aplicação do crescimento ao income.
    next_df['income'] = next_df['income'] * (1 + next_df['g_total'])
    next_df['year'] = year

    projections.append(next_df[['municipio', 'year', 'income']])

    current_df = next_df

projection_df = pd.concat(projections, ignore_index=True)

final_df = pd.concat(
    [df[['municipio', 'year', 'income']], projection_df],
    ignore_index=True
).sort_values(['municipio', 'year']).reset_index(drop=True)

final_df.to_parquet('household_income_projection_2026.parquet', index=False)

final_df.head(20)

NameError: name 'adjust_growth' is not defined

## 6) Sanity checks rápidos

Verificação rápida dos resultados: top municípios em 2026 e um exemplo de série temporal para um município específico, como Lisboa.

In [ ]:
# Top 10 municípios em 2026
final_df[final_df['year'] == 2026].sort_values('income', ascending=False).head(10)


,municipio,year,income
1655,Lisboa,2026,35752.970262
2279,Oeiras,2026,34132.451252
95,Alcochete,2026,34058.626523
2699,Porto,2026,33256.328597
899,Cascais,2026,32226.540471
1067,Coimbra,2026,32127.122090
515,Arruda dos Vinhos,2026,31658.233896
1763,Mafra,2026,31584.892878
983,Castro Verde,2026,31484.817206
1679,Loures,2026,31086.833834


In [ ]:
final_df[final_df['municipio'] == 'Lisboa'].sort_values('year')



,municipio,year,income
1644,Lisboa,2015,24337.000000
1645,Lisboa,2016,24794.000000
1646,Lisboa,2017,25169.000000
1647,Lisboa,2018,27777.000000
1648,Lisboa,2019,28372.000000
1649,Lisboa,2020,28072.000000
1650,Lisboa,2021,29082.000000
1651,Lisboa,2022,28940.975638
1652,Lisboa,2023,30366.956411
1653,Lisboa,2024,32689.817390


## 7) Join com municípios-portugal: limpeza de nomes para o join resultar

O parquet de income tem entradas regionais (ex: 'Norte', 'Continente') e alguns municípios
com capitalização diferente do CSV de municípios. Aqui filtramos e normalizamos antes do join.

In [ ]:
# Ler o CSV de municípios de Portugal
munic_csv = pd.read_csv('municipalities-portugal.csv', sep=';')

# Correcções de capitalização: nomes no parquet que diferem do CSV oficial
name_fixes = {
    'Castro Daire':              'Castro daire',
    'Freixo de Espada à Cinta':  'Freixo de Espada À Cinta',
    'Miranda do Douro':          'Miranda do douro',
    'Ponta Delgada':             'Ponta delgada',
    'Vila da Praia da Vitória':  'Praia da Vitória',
}
final_df['municipio'] = final_df['municipio'].replace(name_fixes)

# Conjunto de municípios válidos (do CSV oficial)
valid_municipios = set(munic_csv['Official Name Municipality'].dropna().unique())

# Remover entradas regionais/agregadas que não são municípios
n_before = len(final_df)
final_df_clean = final_df[final_df['municipio'].isin(valid_municipios)].copy()
removed = sorted(set(final_df['municipio']) - valid_municipios)
print(f'Removidas {n_before - len(final_df_clean)} linhas de {len(removed)} entradas não-municipais:')
print(removed)

# Join com o CSV para obter código oficial e tipo
munic_meta = munic_csv[['Official Name Municipality', 'Official Code Municipality', 'Type']].drop_duplicates(
    subset='Official Name Municipality'
)
final_df_joined = final_df_clean.merge(
    munic_meta,
    left_on='municipio',
    right_on='Official Name Municipality',
    how='left'
).drop(columns='Official Name Municipality')

# Verificar se ficaram NaN no código oficial (indica nomes ainda sem match)
unmatched = final_df_joined[final_df_joined['Official Code Municipality'].isna()]['municipio'].unique()
if len(unmatched):
    print('ATENÇÃO - municípios sem match após join:', sorted(unmatched))
else:
    print('Join completo: todos os municípios têm código oficial.')

final_df_joined.head()